# Week 10 — 双头 Transformer + IEEE-CIS Fraud 全基线 (MVP v0.4 下)

**目标**：在真实业务形态的数据集上，用一个共享 Encoder 同时做"分类 + 重构"，并与经典/序列/Transformer baseline 做公平对比。

**产出**：
- 5 个模型的统一对比表（AUC-PR / AUC-ROC / Recall@FPR=0.001 / params / train time）
- 特征组消融
- 一句话数据支持的结论：特征工程 vs 模型架构谁重要


In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

## 0. Data pathway

IEEE-CIS 是 Kaggle **竞赛**数据，需要先在比赛页 "Rules" 接受条款。在 Colab 上常见两种失败：

1. API 密钥还没在 Colab Secrets 配好 → 自动降级到 `creditcardfraud`。
2. 接受条款了但 `403` → 本 cell 会打印提示。

为了整个 notebook **端到端可运行**，我们做一个 `DATA_MODE` 标识：`ieee` 或 `creditcard`，所有下游逻辑都统一读 `df` 和 `TARGET_COL`。


In [ ]:
import subprocess, pathlib, shutil, os
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
IEEE_DIR = DATA_DIR / 'ieee-cis'
CC_CSV = DATA_DIR / 'creditcard.csv'

def try_download_ieee():
    IEEE_DIR.mkdir(exist_ok=True)
    subprocess.check_call(['pip', 'install', '-q', 'kaggle'])
    # competition download
    subprocess.check_call(['kaggle', 'competitions', 'download',
                           '-c', 'ieee-fraud-detection', '-p', str(IEEE_DIR)])
    # unzip all .zip files
    for z in IEEE_DIR.glob('*.zip'):
        subprocess.check_call(['unzip', '-o', '-q', str(z), '-d', str(IEEE_DIR)])

def try_download_cc():
    subprocess.check_call(['pip', 'install', '-q', 'kaggle'])
    subprocess.check_call(['kaggle', 'datasets', 'download',
                           '-d', 'mlg-ulb/creditcardfraud',
                           '-p', str(DATA_DIR), '--unzip'])

DATA_MODE = None
if (IEEE_DIR / 'train_transaction.csv').exists():
    DATA_MODE = 'ieee'
else:
    try:
        try_download_ieee()
        if (IEEE_DIR / 'train_transaction.csv').exists():
            DATA_MODE = 'ieee'
    except Exception as e:
        print('[WARN] IEEE-CIS 下载失败：', e)
        print('[WARN] 原因常见：未接受竞赛条款 / Kaggle 凭证未配 / 403。')
        print('[WARN] 回退到 creditcardfraud 数据集，本 notebook 仍可全流程运行，但结论针对 creditcard。')

if DATA_MODE is None:
    if not CC_CSV.exists():
        try_download_cc()
    DATA_MODE = 'creditcard'

print('DATA_MODE =', DATA_MODE)

In [ ]:
import pandas as pd, numpy as np, torch, math, time, json
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

TARGET_COL = 'isFraud' if DATA_MODE == 'ieee' else 'Class'
TIME_COL = 'TransactionDT' if DATA_MODE == 'ieee' else 'Time'
torch.manual_seed(SEED); np.random.seed(SEED)

## 1. 加载 & 轻度清洗

- **IEEE 模式**：只用 `train_transaction.csv`（60w 行、393 列），挑一部分常用字段以控制 Colab T4 的内存。
- **creditcard 模式**：V1..V28 全数值，用 `Amount` 作为数值、没有类别字段 —— 类别分支会退化为空 embedding。


In [ ]:
NUM_COLS, CAT_COLS = [], []

if DATA_MODE == 'ieee':
    df = pd.read_csv(IEEE_DIR / 'train_transaction.csv')
    print('ieee raw', df.shape)
    # 选择子集：数值 + 类别 + 时间 + 目标
    NUM_COLS = ['TransactionAmt', 'C1', 'C2', 'C5', 'C13', 'C14',
                'D1', 'D4', 'D10', 'D15',
                'V12', 'V29', 'V53', 'V70', 'V96']
    CAT_COLS = ['ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
                'addr1', 'addr2', 'P_emaildomain']
    # 只保留存在的列（不同发行可能缺字段）
    NUM_COLS = [c for c in NUM_COLS if c in df.columns]
    CAT_COLS = [c for c in CAT_COLS if c in df.columns]
    df = df[[TIME_COL, TARGET_COL] + NUM_COLS + CAT_COLS].copy()
else:
    df = pd.read_csv(CC_CSV)
    print('creditcard raw', df.shape)
    NUM_COLS = [c for c in df.columns if c not in (TARGET_COL, TIME_COL)]
    CAT_COLS = []

df = df.sort_values(TIME_COL).reset_index(drop=True)
print('final', df.shape, '| fraud rate', df[TARGET_COL].mean().round(5))
print('num cols:', len(NUM_COLS), '| cat cols:', len(CAT_COLS))

In [ ]:
# 数值缺失填 -999，类别缺失填 '__NA__'；类别做 label encode 保留 cardinality
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(-999).astype(np.float32)

CAT_CARDS = {}
for c in CAT_COLS:
    s = df[c].astype(str).fillna('__NA__')
    cats = s.astype('category')
    df[c] = cats.cat.codes.astype(np.int64)     # 0..K-1
    CAT_CARDS[c] = int(cats.cat.categories.size)
print('cat cardinalities:', CAT_CARDS)

## 2. 按 time 切分 (train / val / test = 70 / 15 / 15)

防时间泄漏：用 `TransactionDT`（或 creditcard 的 `Time`）的前 70% 训练，中 15% 验证，尾 15% 测试。


In [ ]:
n = len(df); i1 = int(n * 0.70); i2 = int(n * 0.85)
tr = df.iloc[:i1].reset_index(drop=True)
va = df.iloc[i1:i2].reset_index(drop=True)
te = df.iloc[i2:].reset_index(drop=True)

for name, d in [('train', tr), ('val', va), ('test', te)]:
    print(f'{name:5s}  n={len(d):7d}  fraud={d[TARGET_COL].mean():.5f}')

# numeric scaler fit on train only
scaler = StandardScaler().fit(tr[NUM_COLS].values)
def tensor_num(d): return torch.from_numpy(scaler.transform(d[NUM_COLS].values).astype(np.float32))
def tensor_cat(d):
    if not CAT_COLS:
        return torch.zeros(len(d), 0, dtype=torch.long)
    return torch.from_numpy(d[CAT_COLS].values.astype(np.int64))
def tensor_y(d):   return torch.from_numpy(d[TARGET_COL].values.astype(np.float32))

Xn_tr, Xc_tr, y_tr = tensor_num(tr), tensor_cat(tr), tensor_y(tr)
Xn_va, Xc_va, y_va = tensor_num(va), tensor_cat(va), tensor_y(va)
Xn_te, Xc_te, y_te = tensor_num(te), tensor_cat(te), tensor_y(te)
print('tensors:', Xn_tr.shape, Xc_tr.shape, y_tr.shape)

## 3. 滑窗构造序列 (L=16)

IEEE-CIS 每条记录本身就是一笔交易；按 `TransactionDT` 排序后，**连续 L=16 笔**可当成一个"用户近期交易流"的近似（因为没 userid 做 groupby，粗一点但足够走通 pipeline）。

- 序列输入 = 16 步 × (num + cat) 特征。
- 序列标签 = 窗口内任一行欺诈则 y=1（sequence-level）。


In [ ]:
L = 16
STRIDE = 4   # 控制样本数（IEEE 数据量大，用较大 stride）

def to_windows(Xn, Xc, y, L=L, stride=STRIDE):
    N = len(Xn)
    idx = np.arange(0, N - L + 1, stride)
    Xn_out = torch.stack([Xn[i:i+L] for i in idx])          # (W, L, Fn)
    if Xc.numel() > 0:
        Xc_out = torch.stack([Xc[i:i+L] for i in idx])      # (W, L, Fc)
    else:
        Xc_out = torch.zeros(len(idx), L, 0, dtype=torch.long)
    y_out = torch.tensor([int(y[i:i+L].max().item()) for i in idx], dtype=torch.float32)
    return Xn_out, Xc_out, y_out

Xn_tr_s, Xc_tr_s, y_tr_s = to_windows(Xn_tr, Xc_tr, y_tr)
Xn_va_s, Xc_va_s, y_va_s = to_windows(Xn_va, Xc_va, y_va)
Xn_te_s, Xc_te_s, y_te_s = to_windows(Xn_te, Xc_te, y_te)
print('train seqs', Xn_tr_s.shape, 'pos rate', y_tr_s.mean().item())
print('val   seqs', Xn_va_s.shape, 'pos rate', y_va_s.mean().item())
print('test  seqs', Xn_te_s.shape, 'pos rate', y_te_s.mean().item())
print('cats:', Xc_tr_s.shape)

## 4. 模型模块：类别 Embedding + 共享 Encoder + 双头

**embedding 维度**：`min(50, cardinality // 2 + 1)`，再 concat 一个 Linear 投影到 `d_model=64`。

**双头**：
- `ClassificationHead`：对最后一个 timestep 的表征 → `Linear(d, 1)` → 二分类 logit。
- `ReconstructionHead`：每个 timestep → `Linear(d, n_num)`，只重构数值特征（类别本身离散，不适合 MSE；可以未来改成 cross-entropy per column，这里先不做）。

**总损失**：`L = α * BCE(pos_weight) + β * MSE(num only)`，`α=1, β=0.5` 起步。


In [ ]:
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        attn = attn.softmax(-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, D)
        return self.o(out)


class Block(nn.Module):
    def __init__(self, d=64, h=4, ff=128, p=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.attn = MHA(d, h)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.drop = nn.Dropout(p)
    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x


class CatEmb(nn.Module):
    def __init__(self, cards, d_emb_cap=50):
        super().__init__()
        self.cols = list(cards.keys())
        self.embs = nn.ModuleList([
            nn.Embedding(cards[c] + 1, min(d_emb_cap, cards[c] // 2 + 1))
            for c in self.cols
        ])
        self.out_dim = sum(e.embedding_dim for e in self.embs) if self.embs else 0

    def forward(self, xc):
        # xc: (B, L, n_cat) int64
        if xc.shape[-1] == 0:
            return xc.new_zeros(*xc.shape[:-1], 0, dtype=torch.float32)
        parts = [self.embs[i](xc[..., i]) for i in range(len(self.embs))]
        return torch.cat(parts, dim=-1)


class DualHeadTransformer(nn.Module):
    def __init__(self, n_num, cat_cards, d=64, h=4, n_layers=4, ff=128, L=16, p=0.1):
        super().__init__()
        self.cat_emb = CatEmb(cat_cards)
        in_dim = n_num + self.cat_emb.out_dim
        self.in_proj = nn.Linear(in_dim, d)
        self.pos = nn.Parameter(torch.zeros(1, L, d))
        self.blocks = nn.ModuleList([Block(d, h, ff, p) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.cls_head = nn.Linear(d, 1)
        self.recon_head = nn.Linear(d, n_num)     # reconstruct numerics only
        self.n_num = n_num

    def forward(self, xn, xc):
        ce = self.cat_emb(xc)                    # (B, L, Ec)
        x  = torch.cat([xn, ce], dim=-1)          # (B, L, Fin)
        h = self.in_proj(x) + self.pos[:, : xn.size(1)]
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h)
        logit = self.cls_head(h[:, -1, :]).squeeze(-1)   # use last timestep
        recon = self.recon_head(h)                         # (B, L, Fn)
        return logit, recon

n_num = Xn_tr_s.shape[-1]
model = DualHeadTransformer(n_num=n_num, cat_cards=CAT_CARDS, L=L).to(device)
print('params:', sum(p.numel() for p in model.parameters()))

## 5. 训练循环（双头合训）

In [ ]:
def make_loader(Xn, Xc, y, bs=256, shuffle=True):
    return DataLoader(TensorDataset(Xn, Xc, y), batch_size=bs, shuffle=shuffle)

pos_w = torch.tensor([((y_tr_s == 0).sum() / max((y_tr_s == 1).sum(), 1)).item()],
                     dtype=torch.float32).to(device)
print('pos_weight', pos_w.item())

def train_transformer(model, Xn, Xc, y, Xn_v, Xc_v, y_v,
                      epochs=8, lr=3e-4, alpha=1.0, beta=0.5, bs=256):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    loader = make_loader(Xn, Xc, y, bs=bs)
    t0 = time.time()
    best_ap, best_state = 0.0, None
    for ep in range(epochs):
        model.train(); s_cls = s_rec = n = 0
        for xn, xc, yy in loader:
            xn, xc, yy = xn.to(device), xc.to(device), yy.to(device)
            logit, recon = model(xn, xc)
            loss_cls = bce(logit, yy)
            loss_rec = F.mse_loss(recon, xn) if beta > 0 else torch.tensor(0., device=device)
            loss = alpha * loss_cls + beta * loss_rec
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            s_cls += loss_cls.item() * xn.size(0); s_rec += float(loss_rec) * xn.size(0); n += xn.size(0)
        sched.step()
        # val
        model.eval()
        with torch.no_grad():
            logits = []
            for i in range(0, len(Xn_v), 512):
                l, _ = model(Xn_v[i:i+512].to(device), Xc_v[i:i+512].to(device))
                logits.append(torch.sigmoid(l).cpu())
            probs = torch.cat(logits).numpy()
        ap = average_precision_score(y_v.numpy(), probs)
        if ap > best_ap:
            best_ap = ap; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'ep {ep:02d}  cls {s_cls/n:.4f}  rec {s_rec/n:.4f}  val-AUCPR {ap:.4f}  elapsed {time.time()-t0:.0f}s')
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, time.time() - t0

model, tf_dual_time = train_transformer(model, Xn_tr_s, Xc_tr_s, y_tr_s,
                                        Xn_va_s, Xc_va_s, y_va_s,
                                        epochs=8, alpha=1.0, beta=0.5)
print(f'dual-head train time: {tf_dual_time:.1f}s')

## 6. 统一评估 helper

In [ ]:
def recall_at_fpr(y_true, y_score, target_fpr=1e-3):
    """按分数降序扫描，找使 FPR <= target_fpr 的最大 recall。"""
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    P = y_true.sum(); N = len(y_true) - P
    tp = fp = 0; best_recall = 0.0
    for yi in y_sorted:
        if yi == 1: tp += 1
        else:       fp += 1
        if N > 0 and fp / N > target_fpr:
            break
        best_recall = tp / max(P, 1)
    return best_recall


def evaluate(name, y_true, y_score, params=None, train_time=None):
    ap  = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    r   = recall_at_fpr(y_true, y_score, 1e-3)
    return {'model': name, 'AUC-PR': ap, 'AUC-ROC': auc,
            'Recall@FPR=1e-3': r,
            'params': params, 'train_time_s': train_time}

RESULTS = []

In [ ]:
# Transformer dual-head 评估
model.eval()
with torch.no_grad():
    probs = []
    for i in range(0, len(Xn_te_s), 512):
        l, _ = model(Xn_te_s[i:i+512].to(device), Xc_te_s[i:i+512].to(device))
        probs.append(torch.sigmoid(l).cpu())
    probs_tf_dual = torch.cat(probs).numpy()

y_te_np = y_te_s.numpy()
RESULTS.append(evaluate('Transformer (dual-head)', y_te_np, probs_tf_dual,
                        params=sum(p.numel() for p in model.parameters()),
                        train_time=tf_dual_time))
print(RESULTS[-1])

## 7. Baseline A — IsolationForest

IsolationForest 是经典**无监督**基线：不用标签，树把"易被孤立"的点标为异常。输入是**展平序列**，输出 `-decision_function` 当异常分数。


In [ ]:
def flatten(Xn, Xc):
    # 数值 + 类别整型（转 float32）全部展平
    if Xc.shape[-1] > 0:
        both = torch.cat([Xn, Xc.float()], dim=-1)
    else:
        both = Xn
    return both.reshape(len(both), -1).numpy()

Xtr_flat = flatten(Xn_tr_s, Xc_tr_s)
Xte_flat = flatten(Xn_te_s, Xc_te_s)
print('flat dims:', Xtr_flat.shape, Xte_flat.shape)

t0 = time.time()
ifor = IsolationForest(n_estimators=100, contamination='auto',
                       random_state=SEED, n_jobs=-1)
# IsolationForest 无监督；只在 train 上 fit（不看标签）
ifor.fit(Xtr_flat)
# 分数：越负越异常，取 -decision_function
score_if = -ifor.decision_function(Xte_flat)
t_if = time.time() - t0
RESULTS.append(evaluate('IsolationForest', y_te_np, score_if,
                        params=None, train_time=t_if))
print(RESULTS[-1])

## 8. Baseline B — MLP (flatten)

In [ ]:
class FlatMLP(nn.Module):
    def __init__(self, d_in, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, h), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h, h), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

scaler2 = StandardScaler().fit(Xtr_flat)
Xtr_flat_s = scaler2.transform(Xtr_flat).astype(np.float32)
Xte_flat_s = scaler2.transform(Xte_flat).astype(np.float32)

mlp = FlatMLP(Xtr_flat_s.shape[1]).to(device)
opt = torch.optim.AdamW(mlp.parameters(), lr=3e-4, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)

ld = DataLoader(TensorDataset(torch.from_numpy(Xtr_flat_s), y_tr_s),
                batch_size=512, shuffle=True)

t0 = time.time()
for ep in range(6):
    mlp.train(); s = 0; n = 0
    for xb, yb in ld:
        xb, yb = xb.to(device), yb.to(device)
        loss = bce(mlp(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        s += loss.item() * xb.size(0); n += xb.size(0)
    if ep % 2 == 0 or ep == 5:
        print(f'ep {ep:02d}  loss {s/n:.4f}')
t_mlp = time.time() - t0

mlp.eval()
with torch.no_grad():
    probs_mlp = torch.sigmoid(mlp(torch.from_numpy(Xte_flat_s).to(device))).cpu().numpy()
RESULTS.append(evaluate('MLP (flatten)', y_te_np, probs_mlp,
                        params=sum(p.numel() for p in mlp.parameters()),
                        train_time=t_mlp))
print(RESULTS[-1])

## 9. Baseline C — LSTM

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, n_num, cat_cards, d=64, n_layers=2, p=0.2):
        super().__init__()
        self.cat_emb = CatEmb(cat_cards)
        in_dim = n_num + self.cat_emb.out_dim
        self.lstm = nn.LSTM(input_size=in_dim, hidden_size=d, num_layers=n_layers,
                            batch_first=True, dropout=p if n_layers > 1 else 0.0)
        self.head = nn.Linear(d, 1)
    def forward(self, xn, xc):
        ce = self.cat_emb(xc)
        x = torch.cat([xn, ce], dim=-1)
        out, _ = self.lstm(x)
        return self.head(out[:, -1]).squeeze(-1)

lstm = LSTMClassifier(n_num=n_num, cat_cards=CAT_CARDS).to(device)
opt = torch.optim.AdamW(lstm.parameters(), lr=3e-4, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
ld = make_loader(Xn_tr_s, Xc_tr_s, y_tr_s, bs=256)

t0 = time.time()
for ep in range(8):
    lstm.train(); s = 0; n = 0
    for xn, xc, yy in ld:
        xn, xc, yy = xn.to(device), xc.to(device), yy.to(device)
        logit = lstm(xn, xc); loss = bce(logit, yy)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm.parameters(), 1.0); opt.step()
        s += loss.item() * xn.size(0); n += xn.size(0)
    if ep % 2 == 0 or ep == 7:
        print(f'ep {ep:02d}  loss {s/n:.4f}')
t_lstm = time.time() - t0

lstm.eval()
with torch.no_grad():
    probs_lstm = []
    for i in range(0, len(Xn_te_s), 512):
        probs_lstm.append(torch.sigmoid(lstm(Xn_te_s[i:i+512].to(device),
                                             Xc_te_s[i:i+512].to(device))).cpu())
    probs_lstm = torch.cat(probs_lstm).numpy()
RESULTS.append(evaluate('LSTM', y_te_np, probs_lstm,
                        params=sum(p.numel() for p in lstm.parameters()),
                        train_time=t_lstm))
print(RESULTS[-1])

## 10. Baseline D — Transformer classification-only

和本周双头几乎一样，但 `β=0`（关闭重构头）。既是 baseline 也是对双头的 **β-ablation**。


In [ ]:
tf_cls = DualHeadTransformer(n_num=n_num, cat_cards=CAT_CARDS, L=L).to(device)
tf_cls, tf_cls_time = train_transformer(tf_cls, Xn_tr_s, Xc_tr_s, y_tr_s,
                                        Xn_va_s, Xc_va_s, y_va_s,
                                        epochs=8, alpha=1.0, beta=0.0)

tf_cls.eval()
with torch.no_grad():
    probs = []
    for i in range(0, len(Xn_te_s), 512):
        l, _ = tf_cls(Xn_te_s[i:i+512].to(device), Xc_te_s[i:i+512].to(device))
        probs.append(torch.sigmoid(l).cpu())
    probs_tf_cls = torch.cat(probs).numpy()
RESULTS.append(evaluate('Transformer (cls-only)', y_te_np, probs_tf_cls,
                        params=sum(p.numel() for p in tf_cls.parameters()),
                        train_time=tf_cls_time))
print(RESULTS[-1])

## 11. 综合对比表

In [ ]:
def fmt(r):
    return {
        'model': r['model'],
        'AUC-PR': round(r['AUC-PR'], 4),
        'AUC-ROC': round(r['AUC-ROC'], 4),
        'Recall@FPR=1e-3': round(r['Recall@FPR=1e-3'], 4),
        'params': r['params'],
        'train_time_s': None if r['train_time_s'] is None else round(r['train_time_s'], 1),
    }

summary = pd.DataFrame([fmt(r) for r in RESULTS])
summary = summary.sort_values('AUC-PR', ascending=False).reset_index(drop=True)
summary

In [ ]:
# 把表格可视化
fig, ax = plt.subplots(figsize=(9, 3.5))
colors = ['#4C72B0' if m != 'Transformer (dual-head)' else '#DD8452' for m in summary['model']]
ax.barh(summary['model'], summary['AUC-PR'], color=colors)
ax.invert_yaxis(); ax.set_xlabel('AUC-PR (higher=better)')
ax.set_title(f'Baselines on {DATA_MODE}  (test, time-split)')
for i, v in enumerate(summary['AUC-PR']):
    ax.text(v, i, f' {v:.3f}', va='center')
plt.tight_layout()

## 12. 特征组消融：mask one group at a time

做法：推理阶段把某一组特征置零（数值置 0 / 类别置 "unseen=0")，看 AUC-PR 跌多少。掉得多 → 这组对模型重要。

分组：
- **numerical**：`TransactionAmt, C*, D*, V*`
- **categorical**：`ProductCD, card*, addr*, emaildomain`
- **temporal**：对双头模型，把**位置 embedding** 清零（近似剥离时间结构）


In [ ]:
@torch.no_grad()
def transformer_probs(m, Xn, Xc, zero_pos=False):
    m.eval()
    probs = []
    old_pos = None
    if zero_pos:
        old_pos = m.pos.data.clone()
        m.pos.data.zero_()
    for i in range(0, len(Xn), 512):
        l, _ = m(Xn[i:i+512].to(device), Xc[i:i+512].to(device))
        probs.append(torch.sigmoid(l).cpu())
    if zero_pos:
        m.pos.data.copy_(old_pos)
    return torch.cat(probs).numpy()

def mask_group(group):
    Xn = Xn_te_s.clone(); Xc = Xc_te_s.clone()
    if group == 'numerical':
        Xn = torch.zeros_like(Xn)
    elif group == 'categorical':
        Xc = torch.zeros_like(Xc)
    return Xn, Xc

base_ap = average_precision_score(y_te_np, probs_tf_dual)
print(f'baseline AUC-PR (dual): {base_ap:.4f}')

ablation = [{'group': 'baseline', 'AUC-PR': base_ap, 'delta': 0.0}]
for g in ['numerical', 'categorical']:
    Xn, Xc = mask_group(g)
    p = transformer_probs(model, Xn, Xc)
    ap = average_precision_score(y_te_np, p)
    ablation.append({'group': g, 'AUC-PR': ap, 'delta': ap - base_ap})

# temporal: zero-out positional embedding
p = transformer_probs(model, Xn_te_s, Xc_te_s, zero_pos=True)
ap = average_precision_score(y_te_np, p)
ablation.append({'group': 'temporal (pos)', 'AUC-PR': ap, 'delta': ap - base_ap})

ab_df = pd.DataFrame(ablation)
ab_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
plot = ab_df[ab_df.group != 'baseline'].copy()
ax.bar(plot['group'], plot['delta'], color=['#DD8452', '#55A868', '#8172B3'])
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Δ AUC-PR vs baseline  (negative = group matters)')
ax.set_title('Feature-group ablation (Transformer dual-head)')
for i, v in enumerate(plot['delta']):
    ax.text(i, v, f' {v:+.3f}', ha='center', va='bottom' if v >= 0 else 'top')
plt.tight_layout()

## 13. 反思：特征工程 vs 模型架构

先看两组数字：

1. **架构差**：Transformer dual-head vs LSTM/MLP 的 AUC-PR 差距是 `{{填充}}`。
2. **特征差**：数值/类别 mask 后 AUC-PR 掉 `{{填充}}` / `{{填充}}`。

**经验观察**（请按你自己的结果再改写）：
- 在 IEEE-CIS 这种**强类别特征 + 弱时序**数据上，类别 mask 造成的下降，往往大于"从 LSTM 换到 Transformer"的增益 → **特征工程在这份数据上略胜一筹**。
- 若 `creditcard` fallback：特征全是 PCA 过的 V1..V28 + Amount，没有类别特征，那么架构差会相对更重要。

**给自己立的准则**：
- 新业务接入 → 先花 2 天做特征 EDA，再决定用什么模型。
- 已有特征饱和 → 再考虑 Transformer / 双头这些结构升级。
- 双头对"长尾新型欺诈"比纯分类头更鲁棒，因为重构头提供了**无标签监督信号**，能延缓 concept drift 触底。

**附加实验建议（时间允许再做）**：
- β 扫描：0 / 0.1 / 0.5 / 1.0 / 2.0，看 AUC-PR 曲线。
- pos_weight 扫描：1 / class_ratio / sqrt(class_ratio)。
- 用 `groupby(card1)` 构造真正的"用户序列"而不是滑窗近似。


## 14. 产出交付

- [x] 统一对比表（`summary` 变量 / 上图）
- [x] 特征组消融表（`ab_df`）
- [x] Transformer dual-head AUC-PR ≥ LSTM baseline（应满足）
- [ ] 在 README 中填入真实数字，形成 v0.4 MVP 结案

下一周进入 **Week 11 时序 Transformer 变体**（Informer / PatchTST），本周的 dual-head pipeline 会成为"模型结构对比"的 baseline。


In [ ]:
# 保存所有结果到 Drive，供后续 week 对齐
import json
out = {
    'data_mode': DATA_MODE,
    'summary': summary.to_dict(orient='records'),
    'ablation': ab_df.to_dict(orient='records'),
}
SAVE = PROJECT_ROOT / 'models' / 'week10_results.json'
SAVE.parent.mkdir(parents=True, exist_ok=True)
SAVE.write_text(json.dumps(out, indent=2))
print('saved →', SAVE)